In [1]:
import os
import pandas as pd
import numpy as np

In [2]:
DATA_DIR = "../data/processed/pre_processed/"
OUTPUT_DIR = "../data/processed/master/"
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [3]:
customers = pd.read_csv(os.path.join(DATA_DIR, "customers_preprocessed.csv"), parse_dates=['signup_date'])
geography = pd.read_csv(os.path.join(DATA_DIR, "geography_preprocessed.csv"))
inventory = pd.read_csv(os.path.join(DATA_DIR, "inventory_preprocessed.csv"))
order_items = pd.read_csv(os.path.join(DATA_DIR, "order_items_preprocessed.csv"))
orders = pd.read_csv(os.path.join(DATA_DIR, "orders_preprocessed.csv"), parse_dates=['order_date'])
payments = pd.read_csv(os.path.join(DATA_DIR, "payments_preprocessed.csv"))
products = pd.read_csv(os.path.join(DATA_DIR, "products_preprocessed.csv"))
promotions = pd.read_csv(os.path.join(DATA_DIR, "promotions_preprocessed.csv"))
returns = pd.read_csv(os.path.join(DATA_DIR, "returns_preprocessed.csv"))
reviews = pd.read_csv(os.path.join(DATA_DIR, "reviews_preprocessed.csv"))
sales = pd.read_csv(os.path.join(DATA_DIR, "sales_preprocessed.csv"))
shipments = pd.read_csv(os.path.join(DATA_DIR, "shipments_preprocessed.csv"))
web_traffic = pd.read_csv(os.path.join(DATA_DIR, "web_traffic_preprocessed.csv"))

### **1. Tạo bảng `master_sales_detail`**

In [4]:
# Ghi lại số dòng gốc của bảng Base
BASE = len(order_items)
original_rows = len(order_items)
print(f"Số dòng gốc (order_items): {{BASE:,}} dòng")

Số dòng gốc (order_items): {BASE:,} dòng


#### **Gộp dữ liệu**

Step 1 + orders

In [5]:
# ── STEP 1: + orders ─────────────────────────────────────────────
orders_slim = orders[['order_id','order_date','customer_id','zip',
                       'order_status','payment_method','device_type',
                       'order_source','year','month','quarter',
                       'day_of_week','is_weekend']]

master_sales_detail = order_items.merge(orders_slim, on='order_id', how='left')
assert len(master_sales_detail) == BASE, "Cảnh báo: nổ dòng!"
assert master_sales_detail['order_date'].isnull().sum() == 0
print(f"Step 1 + orders:    {len(master_sales_detail):,} rows")

Step 1 + orders:    714,669 rows


Step 2 + geography

In [6]:
# ── STEP 2: + geography (qua orders.zip) ─────────────────────────
# Join geography TRƯỚC customers để tránh conflict cột zip
geo_slim = geography[['zip', 'region']].drop_duplicates('zip')
master_sales_detail = master_sales_detail.merge(geo_slim, on='zip', how='left')
assert len(master_sales_detail) == BASE, "Step 2: nổ dòng!"
assert master_sales_detail['region'].isnull().sum() == 0
print(f"Step 2 + geography: {len(master_sales_detail):,} rows")

Step 2 + geography: 714,669 rows


Step 3 + products

In [7]:
# ── STEP 3: + products ───────────────────────────────────────────
products_slim = products[['product_id','product_name','category','segment',
                           'size','color','price','cogs',
                           'profit_margin','price_tier']]

master_sales_detail = master_sales_detail.merge(products_slim, on='product_id', how='left')
assert len(master_sales_detail) == BASE, "Step 3: nổ dòng!"
assert master_sales_detail['category'].isnull().sum() == 0
print(f"Step 3 + products:  {len(master_sales_detail):,} rows")

Step 3 + products:  714,669 rows


Step 4 + customers

In [8]:
# ── STEP 4: + customers (bỏ zip, city tránh conflict) ────────────
customers_slim = customers[['customer_id','gender','age_group',
                             'acquisition_channel','signup_date',
                             'signup_year','signup_month','signup_cohort']]

master_sales_detail = master_sales_detail.merge(customers_slim, on='customer_id', how='left')
assert len(master_sales_detail) == BASE, "Step 4: nổ dòng!"
assert master_sales_detail['gender'].isnull().sum() == 0
print(f"Step 4 + customers: {len(master_sales_detail):,} rows")

Step 4 + customers: 714,669 rows


Step 5 + payments

In [9]:
# ── STEP 5: + payments ───────────────────────────────────────────
payments_slim = payments[['order_id','installments','is_installment_plan']]

master_sales_detail = master_sales_detail.merge(payments_slim, on='order_id', how='left')
assert len(master_sales_detail) == BASE, "Step 5: nổ dòng!"
assert master_sales_detail['installments'].isnull().sum() == 0
print(f"Step 5 + payments:  {len(master_sales_detail):,} rows")

Step 5 + payments:  714,669 rows


Step 6 + promotions

In [10]:
# ── STEP 6: + promotions (promo_id + promo_id_2) ─────────────────
# Chỉ lấy cột cần thiết, tránh đưa start_date/end_date gây nhầm lẫn
promo_slim = promotions[['promo_id','promo_type','discount_value',
                          'applicable_category','promo_channel',
                          'stackable_flag','promo_duration_days']]

# Join promo_id (primary)
master_sales_detail = master_sales_detail.merge(
    promo_slim.rename(columns={c: f'promo1_{c}' for c in promo_slim.columns
                                if c != 'promo_id'}),
    on='promo_id', how='left'
)

# Join promo_id_2 (secondary)
master_sales_detail = master_sales_detail.merge(
    promo_slim.rename(columns={'promo_id': 'promo_id_2',
                                **{c: f'promo2_{c}' for c in promo_slim.columns
                                   if c != 'promo_id'}}),
    on='promo_id_2', how='left'
)

assert len(master_sales_detail) == BASE, "Step 6: nổ dòng!"
print(f"Step 6 + promotions: {len(master_sales_detail):,} rows")

Step 6 + promotions: 714,669 rows


Step 7 + shipments

In [11]:
# 1. Tính tổng doanh thu của mỗi đơn hàng (Order Level Revenue)
order_totals = master_sales_detail.groupby('order_id')['net_revenue'].sum().reset_index(name='order_total_revenue')

# 2. Ghép tổng doanh thu đơn hàng vào bảng Master
master_sales_detail = master_sales_detail.merge(order_totals, on='order_id', how='left')

# 3. Tính "Tỷ trọng đóng góp" của từng món hàng trong đơn (Từ 0.0 đến 1.0)
# Xử lý ngoại lệ: Nếu tổng đơn = 0 (đơn tặng 100%), thì chia đều tỷ trọng cho số lượng món hàng
item_counts = master_sales_detail.groupby('order_id')['product_id'].transform('count')

master_sales_detail['revenue_share'] = np.where(
    master_sales_detail['order_total_revenue'] == 0,
    1.0 / item_counts,  # Chia đều nếu tổng bill = 0
    master_sales_detail['net_revenue'] / master_sales_detail['order_total_revenue']
)

# 4. Gộp bảng Shipments vào
shipments_slim = shipments[['order_id', 'shipping_fee', 'transit_time_days', 'is_free_shipping']]
master_sales_detail = master_sales_detail.merge(shipments_slim, on='order_id', how='left')

# 5. THỰC HIỆN PHÂN BỔ (ALLOCATION)
master_sales_detail['allocated_shipping_fee'] = master_sales_detail['shipping_fee'] * master_sales_detail['revenue_share']

# Xóa các cột tạm để bảng gọn gàng
master_sales_detail.drop(columns=['order_total_revenue', 'revenue_share'], inplace=True)

assert len(master_sales_detail) == BASE, "Step 7: nổ dòng!"
print(f"Step 6 + shipments: {len(master_sales_detail):,} rows")

Step 6 + shipments: 714,669 rows


#### **Kiểm tra toàn vẹn dữ liệu**

In [12]:
print(f"Số dòng hiện tại: {len(master_sales_detail)}")

if len(master_sales_detail) == BASE:
    print("Thành công: Không bị nổ dòng (Row Explosion).")
else:
    print(f"Cảnh báo: Số dòng bị lệch {len(master_sales_detail) - BASE} dòng!")

Số dòng hiện tại: 714669
Thành công: Không bị nổ dòng (Row Explosion).


In [13]:
master_sales_detail.head()

,order_id,product_id,quantity,unit_price,discount_amount,promo_id,promo_id_2,gross_revenue,net_revenue,is_discounted,...,promo2_promo_type,promo2_discount_value,promo2_applicable_category,promo2_promo_channel,promo2_stackable_flag,promo2_promo_duration_days,shipping_fee,transit_time_days,is_free_shipping,allocated_shipping_fee
0,1,2400,7,1138.22,0.0,NONE,NONE,7967.54,7967.54,0,...,NaN,NaN,NaN,NaN,NaN,NaN,1.37,4.0,0.0,1.37
1,2,609,7,10166.25,0.0,NONE,NONE,71163.75,71163.75,0,...,NaN,NaN,NaN,NaN,NaN,NaN,2.60,4.0,0.0,2.60
2,3,396,3,11220.33,0.0,NONE,NONE,33660.99,33660.99,0,...,NaN,NaN,NaN,NaN,NaN,NaN,2.38,3.0,0.0,2.38
3,4,635,5,10639.25,0.0,NONE,NONE,53196.25,53196.25,0,...,NaN,NaN,NaN,NaN,NaN,NaN,2.49,6.0,0.0,2.49
4,6,1935,1,1597.84,0.0,NONE,NONE,1597.84,1597.84,0,...,NaN,NaN,NaN,NaN,NaN,NaN,25.79,7.0,0.0,25.79


In [14]:
master_sales_detail.info()

<class 'pandas.DataFrame'>
RangeIndex: 714669 entries, 0 to 714668
Data columns (total 58 columns):
 #   Column                      Non-Null Count   Dtype         
---  ------                      --------------   -----         
 0   order_id                    714669 non-null  int64         
 1   product_id                  714669 non-null  int64         
 2   quantity                    714669 non-null  int64         
 3   unit_price                  714669 non-null  float64       
 4   discount_amount             714669 non-null  float64       
 5   promo_id                    714669 non-null  str           
 6   promo_id_2                  714669 non-null  str           
 7   gross_revenue               714669 non-null  float64       
 8   net_revenue                 714669 non-null  float64       
 9   is_discounted               714669 non-null  int64         
 10  discount_percent            714669 non-null  float64       
 11  order_date                  714669 non-null  datet

Kiểm tra dữ liệu thiếu

In [15]:
missing = master_sales_detail.isnull().sum()
missing_pct = missing / len(master_sales_detail) * 100
missing_master_sales_details = pd.DataFrame({'Missing Values': missing, 'Percentage (%)': missing_pct.round(2)})
missing_master_sales_details

,Missing Values,Percentage (%)
order_id,0,0.00
product_id,0,0.00
quantity,0,0.00
unit_price,0,0.00
discount_amount,0,0.00
promo_id,0,0.00
promo_id_2,0,0.00
gross_revenue,0,0.00
net_revenue,0,0.00
is_discounted,0,0.00


Kiểm tra số sản phẩm không sử dụng khuyến mãi trong bảng `order_items`

In [16]:
# ── Kiểm tra tính hợp lệ của Null trong order_items ────────────

# 1. promo1_* null = dòng không có khuyến mãi nào (promo_id = 'NONE')
n_no_promo1 = (master_sales_detail['promo_id'] == 'NONE').sum()
n_null_promo1 = master_sales_detail['promo1_promo_type'].isnull().sum()

print("=== KIỂM TRA promo1_* ===")
print(f"Số dòng promo_id = 'NONE':        {n_no_promo1:,}")
print(f"Số dòng null ở promo1_promo_type: {n_null_promo1:,}")
assert n_no_promo1 == n_null_promo1, "Null promo1 KHÔNG khớp với NONE!"
print("✅ Khớp hoàn toàn → Null promo1_* hợp lệ\n")

# 2. promo2_* null = dòng không có khuyến mãi thứ hai (promo_id_2 = 'NONE')
n_no_promo2 = (master_sales_detail['promo_id_2'] == 'NONE').sum()
n_null_promo2 = master_sales_detail['promo2_promo_type'].isnull().sum()

print("=== KIỂM TRA promo2_* ===")
print(f"Số dòng promo_id_2 = 'NONE':      {n_no_promo2:,}")
print(f"Số dòng null ở promo2_promo_type: {n_null_promo2:,}")
assert n_no_promo2 == n_null_promo2, "Null promo2 KHÔNG khớp với NONE!"
print("✅ Khớp hoàn toàn → Null promo2_* hợp lệ\n")

# 3. Phân phối tổng hợp
total = len(master_sales_detail)
print("=== PHÂN PHỐI KHUYẾN MÃI ===")
both_none  = ((master_sales_detail['promo_id'] == 'NONE') & (master_sales_detail['promo_id_2'] == 'NONE')).sum()
only_promo1 = ((master_sales_detail['promo_id'] != 'NONE') & (master_sales_detail['promo_id_2'] == 'NONE')).sum()
both_promo  = ((master_sales_detail['promo_id'] != 'NONE') & (master_sales_detail['promo_id_2'] != 'NONE')).sum()

print(f"Không có KM nào:      {both_none:,}  ({both_none/total*100:.2f}%)")
print(f"Chỉ có 1 KM (promo1): {only_promo1:,}  ({only_promo1/total*100:.2f}%)")
print(f"Có 2 KM đồng thời:    {both_promo:,}  ({both_promo/total*100:.2f}%)")
print(f"Tổng:                  {both_none+only_promo1+both_promo:,}")

=== KIỂM TRA promo1_* ===
Số dòng promo_id = 'NONE':        438,353
Số dòng null ở promo1_promo_type: 438,353
✅ Khớp hoàn toàn → Null promo1_* hợp lệ

=== KIỂM TRA promo2_* ===
Số dòng promo_id_2 = 'NONE':      714,463
Số dòng null ở promo2_promo_type: 714,463
✅ Khớp hoàn toàn → Null promo2_* hợp lệ

=== PHÂN PHỐI KHUYẾN MÃI ===
Không có KM nào:      438,353  (61.34%)
Chỉ có 1 KM (promo1): 276,110  (38.63%)
Có 2 KM đồng thời:    206  (0.03%)
Tổng:                  714,669


- Số lượng missing values ở các cột `promo1_*` và `promo2_*` khớp hoàn toàn với số lượng sản phẩm không áp dụng khuyến mãi trong bảng `order_items`. Điều này còn cho thấy việc gộp bảng là đúng logic kỹ thuật.

- Ta tiến hành xử lý các missing values bằng phương pháp điền khuyết:
    + Điền "No Promotion" đối với các cột định danh.
    + Điền 0 với các cột định lượng.

--> Việc điền các giá trị này phục vụ cho việc phân tích khám phá.

In [17]:
promo1_text = ['promo1_promo_type','promo1_applicable_category','promo1_promo_channel']
promo1_num  = ['promo1_discount_value','promo1_stackable_flag','promo1_promo_duration_days']
promo2_text = ['promo2_promo_type','promo2_applicable_category','promo2_promo_channel']
promo2_num  = ['promo2_discount_value','promo2_stackable_flag','promo2_promo_duration_days']

master_sales_detail[promo1_text] = master_sales_detail[promo1_text].fillna('No Promotion')
master_sales_detail[promo1_num]  = master_sales_detail[promo1_num].fillna(0)
master_sales_detail[promo2_text] = master_sales_detail[promo2_text].fillna('No Promotion')
master_sales_detail[promo2_num]  = master_sales_detail[promo2_num].fillna(0)

print("\nFillna hoàn tất — không còn null ở các cột promo")
print(f"Null còn lại: {master_sales_detail[promo1_text+promo1_num+promo2_text+promo2_num].isnull().sum().sum()}")


Fillna hoàn tất — không còn null ở các cột promo
Null còn lại: 0


Kiểm tra các trạng thái đơn hàng không có thông tin giao hàng

In [18]:
# Left Join để giữ lại toàn bộ đơn hàng và gắn thêm thông tin ship
merged_check = pd.merge(orders, shipments, on='order_id', how='left')

# Lọc ra danh sách các đơn hàng bị thiếu thông tin vận chuyển (Null ở cột ship_date)
missing_shipments = merged_check[merged_check['ship_date'].isnull()]

# hân tích phân phối của order_status trong nhóm bị thiếu thông tin ship
status_analysis = missing_shipments['order_status'].value_counts().reset_index()
status_analysis.columns = ['Trạng thái đơn hàng', 'Số lượng dòng Null']

# Tính tỷ lệ % đóng góp của từng trạng thái vào tổng số dòng Null
total_nulls = len(missing_shipments)
status_analysis['Tỷ lệ trên tổng Null (%)'] = (status_analysis['Số lượng dòng Null'] / total_nulls * 100).round(2)

print(f"--- PHÂN TÍCH NULL SHIPMENT ---")
print(f"Tổng số dòng bị Null shipment: {total_nulls}")
print(status_analysis)

--- PHÂN TÍCH NULL SHIPMENT ---
Tổng số dòng bị Null shipment: 80878
  Trạng thái đơn hàng  Số lượng dòng Null  Tỷ lệ trên tổng Null (%)
0           cancelled               59462                     73.52
1                paid               13577                     16.79
2             created                7275                      9.00
3           delivered                 524                      0.65
4            returned                  29                      0.04
5             shipped                  11                      0.01


In [19]:
# (Tính tỷ trọng doanh thu để phân bổ)
master_sales_detail['order_total_revenue'] = master_sales_detail.groupby('order_id')['net_revenue'].transform('sum')
item_counts = master_sales_detail.groupby('order_id')['product_id'].transform('count')

master_sales_detail['revenue_share'] = np.where(
    master_sales_detail['order_total_revenue'] == 0,
    1.0 / item_counts, 
    master_sales_detail['net_revenue'] / master_sales_detail['order_total_revenue']
)

# XỬ LÝ NULL THEO TỪNG TRẠNG THÁI ĐƠN HÀNG

# 1. Nhóm 'cancelled': Không đi đơn -> Phí ship = 0
mask_cancelled = master_sales_detail['order_status'] == 'cancelled'
master_sales_detail.loc[mask_cancelled, ['shipping_fee', 'allocated_shipping_fee']] = 0


# 2. Nhóm Freeship thực tế (Hợp lệ): is_free = 1 nhưng thiếu log -> Phí = 0
mask_freeship = (master_sales_detail['is_free_shipping'] == 1) & (master_sales_detail['shipping_fee'].isnull())
master_sales_detail.loc[mask_freeship, ['shipping_fee', 'allocated_shipping_fee']] = 0
# Điền median thời gian giao để có số liệu phân tích vận hành
master_sales_detail.loc[mask_freeship, 'transit_time_days'] = master_sales_detail['transit_time_days'].median()

# 3. Nhóm Lỗi Log: Đơn 'delivered/shipped/returned' + thiếu log
mask_error = (
    master_sales_detail['order_status'].isin(['delivered', 'shipped', 'returned']) & 
    master_sales_detail['transit_time_days'].isnull() 
)


# A. Tài chính: Gán bằng 0 (hoặc Null) để không tạo ra tiền ảo
master_sales_detail.loc[mask_error, ['shipping_fee', 'allocated_shipping_fee']] = 0

# B. Vận hành: CHỈ điền thời gian giao hàng trung bình
avg_transit = master_sales_detail['transit_time_days'].mean()
master_sales_detail.loc[mask_error, 'transit_time_days'] = round(avg_transit) 
master_sales_detail.loc[mask_error, 'is_free_shipping'] = 0 # Đánh cờ là Không Freeship

# Dọn dẹp và kiểm tra
master_sales_detail.drop(columns=['order_total_revenue', 'revenue_share'], inplace=True)

missing_after = master_sales_detail[['shipping_fee', 'is_free_shipping','allocated_shipping_fee', 'transit_time_days']].isnull().sum()
print("\nSố lượng Null CÒN LẠI:")
print(missing_after)


Số lượng Null CÒN LẠI:
shipping_fee              22981
is_free_shipping          88654
allocated_shipping_fee    22981
transit_time_days         88654
dtype: int64


- Vì 'paid' và 'created' chưa phát sinh chi phí vận chuyển. Việc giữ nguyên giá trị `Null` giúp phân tách rõ ràng giữa chi phí thực tế và các đơn hàng đang trong luồng vận hành (Pipeline), tránh làm sai lệch các chỉ số trung bình (Averages)
- Với những đơn hàng có trạng thái 'cancelled' ta giữ nguyên giá trị `Null` ở cột `is_free_shipping` để đảm bảo không bị sai lệch khi phân tích.
- `shipping_fee` và `allocated_shipping_fee` Null của 'paid' và 'created',  `is_free_shipping` và `transit_time_days` Null của cancelled, 'paid' và 'created'.

#### **Feature Engineering**

In [20]:
# Gross margin (Biên lợi nhuận gộp) trên từng dòng sản phẩm
master_sales_detail['line_gross_margin'] = (
    (master_sales_detail['net_revenue'] - master_sales_detail['quantity'] * master_sales_detail['cogs'])
    / master_sales_detail['net_revenue'].replace(0, float('nan'))
).round(4)

# Số ngày khách hàng đã đăng ký tại thời điểm đặt hàng
master_sales_detail['customer_age_at_order_days'] = (master_sales_detail['order_date'] - master_sales_detail['signup_date']).dt.days

#### **Kiểm tra Outlier**

In [21]:
print("Thống kê mô tả cột 'line_gross_margin'")
print(master_sales_detail['line_gross_margin'].describe())
print(f"Âm: {(master_sales_detail['line_gross_margin'] < 0).sum()}")

print('-'*60)

print("Thống kê mô tả cột 'customer_age_at_order_days'")
print(master_sales_detail['customer_age_at_order_days'].describe())
print(f"Âm: {(master_sales_detail['customer_age_at_order_days'] < 0).sum()}")

Thống kê mô tả cột 'line_gross_margin'
count    714669.000000
mean          0.070844
std           0.268607
min          -1.126400
25%          -0.036500
50%           0.084600
75%           0.256700
max           0.527600
Name: line_gross_margin, dtype: float64
Âm: 191349
------------------------------------------------------------
Thống kê mô tả cột 'customer_age_at_order_days'
count    714669.000000
mean       -909.766348
std        1376.926790
min       -3832.000000
25%       -1950.000000
50%       -1004.000000
75%          28.000000
max        3958.000000
Name: customer_age_at_order_days, dtype: float64
Âm: 531554


#### **Final Check**

In [22]:
print("KIỂM ĐỊNH DỮ LIỆU GỘP (DATA VALIDATION)")

# --- 1. GRAIN TEST (Kiểm tra Nổ dòng) ---
print("\n[1] TEST HẠT DỮ LIỆU (Row Count / Grain)")
base_rows = len(order_items)
master_rows = len(master_sales_detail)

if base_rows == master_rows:
    print(f" ✅ PASS: Số dòng khớp tuyệt đối ({base_rows:,} dòng). Không bị nổ dòng hay mất dữ liệu.")
else:
    print(f" ❌ FAIL: Bị lệch dòng! Bảng gốc có {base_rows:,} dòng, Master có {master_rows:,} dòng.")

KIỂM ĐỊNH DỮ LIỆU GỘP (DATA VALIDATION)

[1] TEST HẠT DỮ LIỆU (Row Count / Grain)
 ✅ PASS: Số dòng khớp tuyệt đối (714,669 dòng). Không bị nổ dòng hay mất dữ liệu.


In [23]:
# --- 2. FINANCIAL TEST (Kiểm tra Dòng tiền) ---
print("[2] TEST BẢO TOÀN CHI PHÍ VẬN CHUYỂN")

# Vì shipments có grain là order level (1 dòng/đơn hàng), trong khi master_sales_detail có grain là order-item level (nhiều dòng/đơn hàng) nên cần kiểm tra lại độ chính xác khi gộp dữ liệu

# Tính tổng phí ship bên bảng Shipments gốc (Chỉ tính các đơn có trong bảng Master để tránh lỗi)
valid_orders = master_sales_detail['order_id'].unique()
original_shipping_total = shipments[shipments['order_id'].isin(valid_orders)]['shipping_fee'].sum()

# Tính tổng phí ship bên bảng Master (đã allocation)
allocated_shipping_total = master_sales_detail['allocated_shipping_fee'].sum()

# Kiểm tra độ lệch (Cho phép lệch siêu nhỏ do làm tròn số thập phân)
diff_shipping = abs(original_shipping_total - allocated_shipping_total)

print(f" - Tổng Phí Ship (Bảng Gốc):    {original_shipping_total:,.2f}")
print(f" - Tổng Phí Ship (Bảng Master): {allocated_shipping_total:,.2f}")

if diff_shipping < 1.0: # Lệch dưới 1 đồng coi như Pass
    print(f" ✅ PASS: Dòng tiền bảo toàn! Sai số do làm tròn: {diff_shipping:.4f}")
else:
    print(f" ❌ FAIL: Dòng tiền bị lệch {diff_shipping:,.2f}! Hãy kiểm tra lại công thức Allocation.")

[2] TEST BẢO TOÀN CHI PHÍ VẬN CHUYỂN
 - Tổng Phí Ship (Bảng Gốc):    2,809,309.66
 - Tổng Phí Ship (Bảng Master): 2,809,309.66
 ✅ PASS: Dòng tiền bảo toàn! Sai số do làm tròn: 0.0000


In [24]:
# --- 3. NULL TEST (Kiểm tra Khóa chính) ---
print("[3] TEST TOÀN VẸN KHÓA CHÍNH (Key Integrity)")
critical_keys = ['order_id', 'product_id', 'customer_id']
null_keys = master_sales_detail[critical_keys].isnull().sum()

if null_keys.sum() == 0:
    print(f" ✅ PASS: Không có giá trị Null nào trong các cột định danh quan trọng.")
else:
    print(f" ❌ FAIL: Phát hiện Null ở khóa chính:")
    print(null_keys[null_keys > 0])

print("\n🎯 KẾT LUẬN: SẴN SÀNG PHÂN TÍCH!" if (base_rows == master_rows and diff_shipping < 1.0 and null_keys.sum() == 0) else "⚠️ VUI LÒNG KIỂM TRA LẠI LỖI!")

[3] TEST TOÀN VẸN KHÓA CHÍNH (Key Integrity)
 ✅ PASS: Không có giá trị Null nào trong các cột định danh quan trọng.

🎯 KẾT LUẬN: SẴN SÀNG PHÂN TÍCH!


#### **Xuất file**

In [25]:
output_path = os.path.join(OUTPUT_DIR, "master_sales_detail.csv")
master_sales_detail.to_csv(output_path, index=False)
print(f"Đã lưu: {output_path}")

Đã lưu: ../data/processed/master/master_sales_detail.csv


### **Tạo bảng Master_Time_Series**

In [26]:
# Chuẩn hóa định dạng ngày
sales['Date'] = pd.to_datetime(sales['Date']).dt.date
master_sales_detail['order_date_only'] = pd.to_datetime(master_sales_detail['order_date']).dt.date
web_traffic['date'] = pd.to_datetime(web_traffic['date']).dt.date

**Bước 1:** Nén đặc trưng từ bảng master_sales_detail

In [27]:
daily_features = master_sales_detail.groupby('order_date_only').agg(
    total_orders_actual=('order_id', 'nunique'),
    total_items_sold=('quantity', 'sum'),
    
    # Lấy phí ship đã được làm sạch và phân bổ hoàn hảo
    total_shipping_cost=('allocated_shipping_fee', 'sum'),
    
    # Tính trung bình biên lợi nhuận gộp trong ngày
    avg_line_margin=('line_gross_margin', 'mean'),
    
    # Đếm số lượng sản phẩm có áp dụng khuyến mãi
    promo_applied_items=('promo1_promo_type', lambda x: (x != 'No Promotion').sum())
).reset_index()

**Bước 2:** Nén bảng Web Traffic

In [28]:
daily_traffic = web_traffic.groupby('date').agg(
    total_sessions=('sessions', 'sum'),
    total_unique_visitors=('unique_visitors', 'sum'),
    total_page_views=('page_views', 'sum'),
    avg_bounce_rate=('bounce_rate', 'mean')
).reset_index()

**Bước 3:** Gộp vào bảng Sales

In [29]:
# Bảng Sales đóng vai trò làm trục xương sống (Ground Truth)
master_time_series = sales.merge(
    daily_features, left_on='Date', right_on='order_date_only', how='left'
).merge(
    daily_traffic, left_on='Date', right_on='date', how='left'
)

# Dọn dẹp cột thừa
master_time_series.drop(columns=['order_date_only', 'date'], inplace=True, errors='ignore')

# Điền 0 cho những ngày không có đơn/không có traffic
# Fillna 0 cho các cột count
count_cols = ['total_orders_actual','total_items_sold','total_shipping_cost',
              'promo_applied_items','total_sessions','total_unique_visitors','total_page_views']
master_time_series[count_cols] = master_time_series[count_cols].fillna(0)

# Fillna mean cho các cột rate
rate_cols = ['avg_bounce_rate', 'avg_line_margin']
master_time_series[rate_cols] = master_time_series[rate_cols].fillna(
    master_time_series[rate_cols].mean()
)

**Bước 4:** Feature Engineering

In [30]:
# 1. Tỷ lệ chuyển đổi (Đơn hàng / Phiên truy cập)
master_time_series['conversion_rate'] = (master_time_series['total_orders_actual'] / master_time_series['total_sessions'].replace(0, np.nan)).fillna(0)

# 2. Giá trị trung bình trên một khách truy cập (Revenue per Visitor)
master_time_series['revenue_per_visitor'] = (master_time_series['Revenue'] / master_time_series['total_unique_visitors'].replace(0, np.nan)).fillna(0)

print(master_time_series.info())


<class 'pandas.DataFrame'>
RangeIndex: 3833 entries, 0 to 3832
Data columns (total 25 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Date                   3833 non-null   object 
 1   Revenue                3833 non-null   float64
 2   COGS                   3833 non-null   float64
 3   is_gross_loss          3833 non-null   int64  
 4   gross_loss_amount      3833 non-null   float64
 5   is_revenue_outlier     3833 non-null   int64  
 6   is_cogs_outlier        3833 non-null   int64  
 7   year                   3833 non-null   int64  
 8   month                  3833 non-null   int64  
 9   quarter                3833 non-null   int64  
 10  day_of_week            3833 non-null   int64  
 11  is_weekend             3833 non-null   int64  
 12  gross_profit           3833 non-null   float64
 13  gross_margin           3833 non-null   float64
 14  total_orders_actual    3833 non-null   int64  
 15  total_items_sol

Kiểm tra logic

In [31]:
print("NGHIỆM THU DỮ LIỆU CHUỖI THỜI GIAN (TIME SERIES AUDIT)")


# --- 1. DATE CONTINUITY TEST ---
# Chuyển Date về dạng datetime để kiểm tra khoảng cách
master_time_series['Date'] = pd.to_datetime(master_time_series['Date'])
date_range = (master_time_series['Date'].max() - master_time_series['Date'].min()).days + 1
actual_days = len(master_time_series)

print("\n[1] TEST TÍNH LIÊN TỤC CỦA THỜI GIAN:")
if date_range == actual_days:
    print(f" ✅ PASS: Chuỗi thời gian liên tục 100% ({actual_days} ngày không bị đứt gãy).")
else:
    print(f" ❌ FAIL: Phát hiện đứt gãy! Khung thời gian có {date_range} ngày nhưng chỉ có {actual_days} dòng dữ liệu.")

NGHIỆM THU DỮ LIỆU CHUỖI THỜI GIAN (TIME SERIES AUDIT)

[1] TEST TÍNH LIÊN TỤC CỦA THỜI GIAN:
 ✅ PASS: Chuỗi thời gian liên tục 100% (3833 ngày không bị đứt gãy).


In [32]:
# --- 2. MACRO FINANCIAL TEST ---
print("\n[2] TEST BẢO TOÀN DÒNG TIỀN VĨ MÔ:")
# A. Check với Sales BTC
original_sales_revenue = sales['Revenue'].sum()
ts_revenue = master_time_series['Revenue'].sum()
diff_revenue = abs(original_sales_revenue - ts_revenue)

if diff_revenue < 1e-6:
    print(f" ✅ PASS: Tổng doanh thu (Revenue) khớp hoàn hảo với nhãn gốc của BTC! ({ts_revenue:,.2f})")
else:
    print(f" ❌ FAIL: Doanh thu bị lệch {diff_revenue:,.2f} so với nhãn gốc!")

# B. Check với Master Detail
original_shipping = master_sales_detail['allocated_shipping_fee'].sum()
ts_shipping = master_time_series['total_shipping_cost'].sum()
diff_shipping = abs(original_shipping - ts_shipping)

if diff_shipping < 1.0:
    print(f" ✅ PASS: Tổng phí vận chuyển khớp hoàn hảo với bảng Master Detail! ({ts_shipping:,.2f})")
else:
    print(f" ❌ FAIL: Phí ship bị rò rỉ trong quá trình Aggregate! Lệch {diff_shipping:,.2f}")


[2] TEST BẢO TOÀN DÒNG TIỀN VĨ MÔ:
 ✅ PASS: Tổng doanh thu (Revenue) khớp hoàn hảo với nhãn gốc của BTC! (16,430,476,585.53)
 ✅ PASS: Tổng phí vận chuyển khớp hoàn hảo với bảng Master Detail! (2,809,309.66)


In [33]:
# --- 3. FEATURE LOGIC TEST ---
print("\n[3] TEST LOGIC BIẾN ĐẶC TRƯNG:")
invalid_conversion = master_time_series[master_time_series['conversion_rate'] > 1.0]
if len(invalid_conversion) == 0:
    print(" ✅ PASS: Tỷ lệ chuyển đổi (Conversion Rate) hợp lý (<= 1.0).")
else:
    print(f" ❌ FAIL: Có {len(invalid_conversion)} ngày tỷ lệ chuyển đổi > 100%! Hãy kiểm tra lại lượt Sessions.")


[3] TEST LOGIC BIẾN ĐẶC TRƯNG:
 ✅ PASS: Tỷ lệ chuyển đổi (Conversion Rate) hợp lý (<= 1.0).


In [34]:
# --- 4. NULL TEST TỔNG THỂ ---
print("\n[4] TEST NULL:")
total_nulls = master_time_series.isnull().sum().sum()
if total_nulls == 0:
    print(" ✅ PASS: Bảng sạch sẽ 100%, không còn bất kỳ giá trị Null nào.")
else:
    print(f" ❌ FAIL: Phát hiện {total_nulls} giá trị Null. Machine Learning model sẽ bị lỗi!")

print("\n" + "="*60)
print("🎯 File đã sẵn sàng để huấn luyện mô hình!" if (date_range==actual_days and diff_revenue<1 and total_nulls==0) else "⚠️ CẦN SỬA LỖI TRƯỚC KHI TRAINING!")


[4] TEST NULL:
 ✅ PASS: Bảng sạch sẽ 100%, không còn bất kỳ giá trị Null nào.

🎯 File đã sẵn sàng để huấn luyện mô hình!


Xuất file

In [35]:
ts_output_path = "../data/processed/master/master_time_series.csv"
master_time_series.to_csv(ts_output_path, index=False)
print(f"\nĐã tạo bảng Master Time Series")


Đã tạo bảng Master Time Series
